# K-Nearest Neighbors (KNN) Classification
**Dataset:** Red Wine Quality (UCI Machine Learning Repository)  
**Goal:** Classify red wines as "good" or "bad" using physicochemical measurements, first by implementing KNN from scratch, then using scikit-learn.

---

## 1. Background: What is K-Nearest Neighbors?

K-Nearest Neighbors (KNN) is a simple but powerful classification algorithm. Rather than learning an explicit model during training, it memorizes the entire dataset and makes predictions at inference time by looking at the closest examples it has seen before.

**How it works:**
1. **Choose K** — a hyperparameter controlling how many neighbors to consult
2. **Calculate distances** — measure how far the new point is from every training point (we'll use Euclidean distance)
3. **Find the K nearest neighbors** — sort by distance and take the top K
4. **Majority vote** — whichever class appears most among those K neighbors wins

Because KNN relies entirely on distance, the scale of your features matters a lot. A feature measured in thousands will dominate one measured in decimals, so normalization is essential before running the algorithm.

## 2. Setup & Data Loading

In [ ]:
# Download the red wine quality dataset from Google Drive
!gdown 1RnmUfRSJWZxM7Ars3N3kTUEOBIJLSfrl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the dataset and take a first look
wine_df = pd.read_csv('winequality-red.csv')
print(f"Shape: {wine_df.shape}")
wine_df.head()

In [ ]:
# Quick summary statistics to understand the feature ranges
# Notice how different the scales are — this is why we need normalization
wine_df.describe()

## 3. Exploratory Data Analysis

Before building the classifier, it's worth visualizing the data. Since KNN is distance-based, we want to understand how the features are distributed and whether there's any obvious separation between good and bad wines.

In [ ]:
# Distribution of wine quality scores
plt.figure(figsize=(8, 4))
wine_df['quality'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribution of Wine Quality Scores')
plt.xlabel('Quality Score')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot of a few key features, colored by quality threshold
# We'll define "good" as quality >= 6 (matches our classification label below)
wine_df['quality_label'] = (wine_df['quality'] >= 6).map({True: 'Good', False: 'Bad'})

sns.pairplot(
    wine_df[['alcohol', 'volatile_acidity', 'sulphates', 'citric_acid', 'quality_label']],
    hue='quality_label',
    palette={'Good': 'steelblue', 'Bad': 'salmon'},
    plot_kws={'alpha': 0.5}
)
plt.suptitle('Pairplot of Key Wine Features by Quality', y=1.02)
plt.show()

## 4. KNN from Scratch

Before reaching for scikit-learn, we'll implement KNN by hand. This helps build intuition for what the algorithm is actually doing under the hood.

### 4.1 Define the unknown wines we want to classify

These three wine samples have known measurements but unknown quality labels — we'll use our KNN to predict whether each is "good" or "bad".

In [ ]:
# Three wine samples we want to classify
# Features: fixed_acidity, volatile_acidity, citric_acid, residual_sugar,
#           chlorides, free_sulfur_dioxide, total_sulfur_dioxide,
#           density, pH, sulphates, alcohol
unknown1 = [12.2, 0.45, 0.49, 1.4,  0.075,  3.0,  6.0, 0.9969,  3.13, 0.63, 10.4]
unknown2 = [8.3,  0.34, 0.4,  2.4,  0.065, 24.0, 48.0, 0.99554, 3.34, 0.86, 11.0]
unknown3 = [8.8,  0.59, 0.18, 2.9,  0.089, 12.0, 74.0, 0.99738, 3.14, 0.54,  9.4]

unknowns_raw = [unknown1, unknown2, unknown3]
print("Unknowns defined. Each has 11 feature values.")

### 4.2 Create binary labels

The original dataset uses quality scores from 3–8. We collapse these into a binary label: **1 = good (quality ≥ 6)**, **0 = bad (quality < 6)**. This simplifies the classification problem.

In [ ]:
# Reload clean copy of the dataframe (without the EDA label column)
wine_df = pd.read_csv('winequality-red.csv')

# Binary classification: good wine = quality score of 6 or higher
wine_df['label'] = (wine_df['quality'] >= 6).astype(int)

print(f"Good wines (label=1): {wine_df['label'].sum()}")
print(f"Bad wines  (label=0): {(wine_df['label'] == 0).sum()}")

### 4.3 Normalize the features

Because KNN is purely distance-based, features with larger numeric ranges (like `total_sulfur_dioxide`, which can be 6–289) will dominate features with smaller ranges (like `pH`, which is ~2.7–4.0). We normalize using **z-score standardization**: subtract the mean and divide by the standard deviation so every feature ends up on a comparable scale.

In [ ]:
features = [
    'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar',
    'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide',
    'density', 'pH', 'sulphates', 'alcohol'
]

X = wine_df[features].values
y = wine_df['label'].values

# Compute mean and std from the training data — save these to apply to unknowns later
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0)

# Normalize
X_scaled = (X - X_mean) / X_std

print("Features normalized. Before vs after for first sample:")
print(f"  Raw:    {X[0]}")
print(f"  Scaled: {X_scaled[0].round(3)}")

### 4.4 Implement Euclidean distance

Euclidean distance is the straight-line distance between two points in n-dimensional space:

$$d = \sqrt{\sum_{i=1}^{n}(a_i - b_i)^2}$$

In [ ]:
def euclidean_distance(a, b):
    """
    Compute the Euclidean distance between two feature vectors.
    
    Parameters
    ----------
    a, b : array-like
        Two vectors of equal length.
    
    Returns
    -------
    float
        The straight-line distance between a and b.
    """
    return np.sqrt(np.sum((np.array(a) - np.array(b)) ** 2))

# Quick sanity check
d = euclidean_distance(X_scaled[0], X_scaled[1])
print(f"Distance between wine 0 and wine 1 (scaled): {d:.4f}")

### 4.5 Implement the KNN classifier

In [ ]:
def knn_predict(X_train, y_train, x_test, k=5):
    """
    Predict the class label for a single test point using KNN.
    
    Parameters
    ----------
    X_train : ndarray of shape (n_samples, n_features)
        Scaled training feature matrix.
    y_train : ndarray of shape (n_samples,)
        Training labels.
    x_test : array-like of shape (n_features,)
        A single scaled test point.
    k : int
        Number of nearest neighbors to use.
    
    Returns
    -------
    int
        Predicted class label (majority vote among k neighbors).
    """
    # Calculate distance from x_test to every training point
    distances = [(euclidean_distance(X_train[i], x_test), y_train[i])
                 for i in range(len(X_train))]
    
    # Sort by distance (ascending) and grab the k closest
    distances.sort(key=lambda x: x[0])
    neighbors = distances[:k]
    
    # Majority vote: return whichever label appears most often
    neighbor_labels = [label for _, label in neighbors]
    return max(set(neighbor_labels), key=neighbor_labels.count)

### 4.6 Classify the unknown wines

Remember to apply the *same* normalization (using the training mean and std) to the unknowns before predicting — otherwise the distances are meaningless.

In [ ]:
# Apply the same scaling used on training data to the unknown wines
unknowns_scaled = [(np.array(u) - X_mean) / X_std for u in unknowns_raw]

k = 5

print(f"Using K = {k}\n")
for i, u in enumerate(unknowns_scaled, 1):
    pred = knn_predict(X_scaled, y, u, k)
    result = "GOOD WINE ✓" if pred == 1 else "BAD WINE ✗"
    print(f"Unknown {i}: {result}")

## 5. KNN with scikit-learn

Now we'll replicate the workflow using scikit-learn, which is what you'd use in practice. It's faster, handles edge cases, and integrates cleanly with other sklearn tools like `train_test_split` and `StandardScaler`.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score,
                              ConfusionMatrixDisplay)

In [ ]:
# Reload and prepare features & labels
wine_df = pd.read_csv('winequality-red.csv')
wine_df['label'] = (wine_df['quality'] >= 6).astype(int)

X = wine_df[features].values
y = wine_df['label'].values

# Split into training (80%) and testing (20%) sets
# random_state ensures reproducibility — the same split every run
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

In [ ]:
# Scale features using only the training data
# IMPORTANT: fit the scaler on X_train only — never on X_test.
# Fitting on test data would leak information and inflate accuracy estimates.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)   # apply same transform, don't refit

In [ ]:
# Train and evaluate KNN with K=5 as a baseline
k = 5
clf = KNeighborsClassifier(n_neighbors=k)
clf.fit(X_train_sc, y_train)

y_pred = clf.predict(X_test_sc)
acc = accuracy_score(y_test, y_pred)
print(f"Baseline accuracy (K={k}): {acc:.4f}")

## 6. Evaluating the Model: Confusion Matrix

Accuracy alone can be misleading — especially with imbalanced datasets. A confusion matrix breaks down predictions into four categories:

|  | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actually Positive** | True Positive (TP) | False Negative (FN) |
| **Actually Negative** | False Positive (FP) | True Negative (TN) |

From these we can compute:
- **Accuracy** = (TP + TN) / Total
- **Precision** = TP / (TP + FP) — *of wines predicted good, how many actually were?*
- **Recall** = TP / (TP + FN) — *of all good wines, how many did we catch?*
- **F1 Score** = 2 × (Precision × Recall) / (Precision + Recall)

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Bad (0)', 'Good (1)'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — KNN (K={k})', fontsize=13)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp)
recall    = tp / (tp + fn)
f1        = 2 * precision * recall / (precision + recall)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

## 7. Choosing the Best K

K is the most important hyperparameter in KNN. 
- **Too small (K=1):** the model memorizes the training data and is very sensitive to noise (overfitting)
- **Too large (K=N):** the model just predicts the majority class for everything (underfitting)

The sweet spot is somewhere in between. We can find it by testing a range of K values and comparing accuracy on the test set.

In [ ]:
# Test accuracy for K = 1 through 30
k_range = range(1, 31)
acc_vals = []

for k in k_range:
    clf_k = KNeighborsClassifier(n_neighbors=k)
    clf_k.fit(X_train_sc, y_train)
    y_pred_k = clf_k.predict(X_test_sc)
    acc_vals.append(accuracy_score(y_test, y_pred_k))

# Plot the results
best_k = k_range[acc_vals.index(max(acc_vals))]

plt.figure(figsize=(10, 5))
plt.plot(k_range, acc_vals, marker='o', linewidth=2, markersize=5, color='steelblue')
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K = {best_k}')
plt.title('Test Accuracy as a Function of K', fontsize=14)
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Best K: {best_k}  |  Accuracy: {max(acc_vals):.4f}")

## 8. Final Predictions on the Unknown Wines

Using our best K value, let's classify the three unknown wine samples with the sklearn pipeline.

In [ ]:
# Scale the unknowns using the same scaler fitted on training data
unknowns_sc = scaler.transform(unknowns_raw)

# Predict with the best K found above
clf_best = KNeighborsClassifier(n_neighbors=best_k)
clf_best.fit(X_train_sc, y_train)

predictions = clf_best.predict(unknowns_sc)

print(f"Predictions (K={best_k}):\n")
for i, pred in enumerate(predictions, 1):
    label_str = "GOOD WINE ✓" if pred == 1 else "BAD WINE ✗"
    print(f"  Unknown {i}: {label_str}")

## 9. Bonus: Apply KNN to a Heart Disease Dataset

Let's apply the same pipeline to a different dataset — heart disease classification — to show that the approach generalizes.

In [ ]:
# Download the heart disease dataset
!gdown 1A7dxaHxMog5rFWiNoORmeFE41cY6AFRT

In [ ]:
heart_df = pd.read_csv('heart.csv')
print(f"Shape: {heart_df.shape}")
heart_df.head()

In [ ]:
# Prepare features and labels
# The target column (1 = heart disease, 0 = no heart disease)
X_h = heart_df.drop(columns=['target']).values
y_h = heart_df['target'].values

# Train/test split and scaling
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

scaler_h = StandardScaler()
X_h_train_sc = scaler_h.fit_transform(X_h_train)
X_h_test_sc  = scaler_h.transform(X_h_test)

In [ ]:
# Sweep K values and find the best one
heart_acc_vals = []

for k in k_range:
    clf_h = KNeighborsClassifier(n_neighbors=k)
    clf_h.fit(X_h_train_sc, y_h_train)
    y_h_pred = clf_h.predict(X_h_test_sc)
    heart_acc_vals.append(accuracy_score(y_h_test, y_h_pred))

best_k_h = k_range[heart_acc_vals.index(max(heart_acc_vals))]

plt.figure(figsize=(10, 5))
plt.plot(k_range, heart_acc_vals, marker='o', linewidth=2, markersize=5, color='darkorange')
plt.axvline(x=best_k_h, color='red', linestyle='--', label=f'Best K = {best_k_h}')
plt.title('Heart Disease Dataset — Accuracy vs. K', fontsize=14)
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Best K: {best_k_h}  |  Accuracy: {max(heart_acc_vals):.4f}")

In [ ]:
# Confusion matrix for best heart disease model
clf_h_best = KNeighborsClassifier(n_neighbors=best_k_h)
clf_h_best.fit(X_h_train_sc, y_h_train)
y_h_pred_best = clf_h_best.predict(X_h_test_sc)

cm_h = confusion_matrix(y_h_test, y_h_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
disp_h = ConfusionMatrixDisplay(confusion_matrix=cm_h,
                                display_labels=['No Disease', 'Disease'])
disp_h.plot(ax=ax, colorbar=False, cmap='Oranges')
ax.set_title(f'Confusion Matrix — Heart Disease KNN (K={best_k_h})', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Summary

In this notebook we:

1. **Implemented KNN from scratch** using only Python and NumPy — computing Euclidean distances, sorting neighbors, and running a majority vote
2. **Replicated the pipeline with scikit-learn**, using `KNeighborsClassifier`, `StandardScaler`, and `train_test_split`
3. **Evaluated the model** using a confusion matrix, accuracy, precision, recall, and F1 score
4. **Tuned K** by sweeping values 1–30 and selecting the one with the highest test accuracy
5. **Applied the same workflow to a second dataset** (heart disease) to demonstrate generalizability

**Key takeaways:**
- Always normalize features before running KNN — unscaled features will skew distance calculations
- The choice of K significantly impacts performance; use a validation sweep rather than guessing
- Accuracy alone can be misleading; always check precision/recall, especially with imbalanced data